# 02. Моделирование и сравнение подходов

Цель ноутбука — сравнить статистические, классические ML- и нейросетевые подходы к прогнозированию средней суточной температуры воздуха в Волгограде на одном и том же датасете.

В доработанной версии исследования акцент сделан на двух вопросах:

- насколько полезно увеличение длины обучающей истории при фиксированных validation и test интервалах;
- дают ли нейросетевые модели устойчивое преимущество по сравнению с аккуратными статистическими и табличными baseline.


## Постановка задачи и защита от утечки информации

В работе рассматривается однодневный прогноз: для даты `t` нужно предсказать `target_tavg`, используя только информацию, доступную не позже конца дня `t-1`.

Чтобы сравнение было честным и воспроизводимым, в проекте фиксируются:

- конечная дата датасета;
- validation-интервал `2024-01-01 ... 2024-12-31`;
- test-интервал `2025-01-01 ... 2025-12-31`.

Во всех сценариях меняется только старт обучающего периода. Это позволяет изолированно оценить влияние объёма train-истории: validation и test остаются одинаковыми для всех моделей и для всех сравнений.

Случайное перемешивание не используется. Лаговые и скользящие признаки уже построены только по прошлым значениям температуры, а суточные метеопризнаки в табличной модели дополнительно сдвигаются на один день назад.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.ar_model import AutoReg
import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.constants import FIGURES_DIR, MODEL_DATASET_FILENAME, PROCESSED_DATA_DIR, TABLES_DIR
from src.utils import (
    build_sequence_features,
    build_tabular_modeling_frame,
    build_train_scenarios,
    create_sequence_windows,
    ensure_project_directories,
    evaluate_forecast,
    get_default_runtime_config,
    get_interval_summary,
    get_yearly_coverage,
    seasonal_naive_forecast,
    set_global_seed,
    split_dataset_by_dates,
    split_sequence_windows_by_dates,
)

ensure_project_directories()
set_global_seed()
tf.get_logger().setLevel("ERROR")
runtime_config = get_default_runtime_config()


## Служебный шаг. Очистка устаревших артефактов второго ноутбука

Ниже удаляются старые файлы `02_*` в `outputs/figures` и `outputs/tables`, которые не относятся к текущей версии проекта. Это помогает не смешивать результаты прошлых экспериментов с новой версией ноутбука. Артефакты первого ноутбука (`01_*`) не затрагиваются.


In [ ]:
CURRENT_02_FIGURES = {
    "02_train_size_mae.png",
    "02_main_test_predictions.png",
    "02_main_test_metrics.png",
    "02_neural_learning_curves.png",
    "02_error_by_season.png",
}
CURRENT_02_TABLES = {
    "02_yearly_coverage.csv",
    "02_train_scenarios.csv",
    "02_train_size_validation_metrics.csv",
    "02_train_size_test_metrics.csv",
    "02_train_size_neural_advantage.csv",
    "02_main_validation_metrics.csv",
    "02_main_test_metrics.csv",
    "02_main_test_predictions.csv",
    "02_random_forest_timeseriessplit.csv",
    "02_seasonal_error_analysis.csv",
}

removed_outputs = []
for output_path in sorted(FIGURES_DIR.glob("02_*")):
    if output_path.name not in CURRENT_02_FIGURES:
        output_path.unlink(missing_ok=True)
        removed_outputs.append(str(output_path))

for output_path in sorted(TABLES_DIR.glob("02_*")):
    if output_path.name not in CURRENT_02_TABLES:
        output_path.unlink(missing_ok=True)
        removed_outputs.append(str(output_path))

if removed_outputs:
    print("Удалены устаревшие артефакты предыдущих версий ноутбука 02:")
    for item in removed_outputs:
        print(f"- {item}")
else:
    print("Устаревших артефактов для удаления не найдено.")


## Шаг 1. Загрузка подготовленного датасета

Этот ноутбук предполагает, что файл `data/processed/volgograd_daily_temperature_dataset.csv` уже создан первым ноутбуком. Для воспроизводимости ниже дополнительно фиксируется конечная дата анализа: даже если в CSV окажутся более поздние строки, в эксперимент они не попадут.


In [ ]:
dataset_path = PROCESSED_DATA_DIR / MODEL_DATASET_FILENAME
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Файл {dataset_path} не найден. Сначала запустите notebooks/01_dataset_creation.ipynb."
    )

dataset_raw = pd.read_csv(dataset_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
dataset = dataset_raw.loc[
    (dataset_raw["date"] >= runtime_config["data_start_date"])
    & (dataset_raw["date"] <= runtime_config["data_end_date"])
].copy().reset_index(drop=True)

print(f"Загружен файл: {dataset_path}")
print(
    "Период в исходном CSV: "
    f"{dataset_raw['date'].min().date()} ... {dataset_raw['date'].max().date()}"
)
print(
    "Период, используемый в исследовании: "
    f"{dataset['date'].min().date()} ... {dataset['date'].max().date()}"
)
print(f"Число строк после фиксации периода: {len(dataset):,}")
display(dataset.head())


## Шаг 2. Проверка временного покрытия и сценарии длины train

Перед сравнением моделей полезно убедиться, что ранние годы действительно имеют достаточное покрытие. Для этого считаем число наблюдений по годам и строим несколько сценариев длины train.

В качестве самого длинного сценария берётся не просто максимально ранняя дата, а **максимум качественной истории**: самый ранний полный год до validation-периода, который проходит порог по полноте наблюдений.


In [ ]:
coverage = get_yearly_coverage(dataset)
scenarios = build_train_scenarios(dataset)

scenario_rows = []
for scenario in scenarios:
    train_days = int((scenario["train_end"] - scenario["train_start"]).days + 1)
    scenario_rows.append(
        {
            "scenario": scenario["scenario"],
            "train_start": scenario["train_start"].date(),
            "train_end": scenario["train_end"].date(),
            "train_days": train_days,
            "train_years_approx": round(train_days / 365.25, 2),
            "description": scenario["description"],
        }
    )

scenarios_df = pd.DataFrame(scenario_rows)

coverage_path = TABLES_DIR / "02_yearly_coverage.csv"
scenarios_path = TABLES_DIR / "02_train_scenarios.csv"
coverage.to_csv(coverage_path, index=False)
scenarios_df.to_csv(scenarios_path, index=False)

display(Markdown("### Годовое покрытие датасета"))
display(coverage.round(3))

display(Markdown("### Сценарии длины обучающей истории"))
display(scenarios_df)

first_partial_year = coverage.loc[coverage["share_of_full_year"] < 0.98, "year"]
if not first_partial_year.empty:
    print(
        "Самые ранние частично заполненные годы не используются как старт максимального train, "
        "если они не проходят порог полноты."
    )

print(f"Таблица покрытия сохранена в: {coverage_path}")
print(f"Таблица сценариев сохранена в: {scenarios_path}")


## Шаг 3. Подготовка признаков для разных классов моделей

В проекте используются два представления данных:

- **табличное** — для baseline, статистической модели, `RandomForest` и `Dense`-сети; здесь применяются лаги, скользящие статистики и календарные признаки;
- **последовательностное** — для `GRU` и `Conv1D`; здесь модель получает окно из последних 30 дней и сама извлекает полезную динамику из последовательности.

Такой набор моделей усиливает тему курсовой именно про нейросети: Dense-сеть сравнивается с классическим табличным ML, а GRU и Conv1D показывают, помогает ли явная обработка временного окна.


In [ ]:
tabular_df, feature_columns, weather_columns = build_tabular_modeling_frame(dataset)
sequence_feature_columns = build_sequence_features(dataset)
X_sequence, y_sequence, sequence_dates = create_sequence_windows(
    dataset,
    sequence_feature_columns,
    window_size=runtime_config["sequence_window_days"],
)

scenario_lookup = {scenario["scenario"]: scenario for scenario in scenarios}
main_scenario = scenario_lookup["Максимум качественной истории"]

train_df, validation_df, test_df = split_dataset_by_dates(
    tabular_df,
    train_start=main_scenario["train_start"],
    validation_start=runtime_config["validation_start_date"],
    validation_end=runtime_config["validation_end_date"],
    test_start=runtime_config["test_start_date"],
    test_end=runtime_config["test_end_date"],
)

sequence_main = split_sequence_windows_by_dates(
    X_sequence,
    y_sequence,
    sequence_dates,
    train_start=main_scenario["train_start"],
    validation_start=runtime_config["validation_start_date"],
    validation_end=runtime_config["validation_end_date"],
    test_start=runtime_config["test_start_date"],
    test_end=runtime_config["test_end_date"],
)

validation_sequence_dates = pd.Series(pd.to_datetime(sequence_main["dates_validation"]))
test_sequence_dates = pd.Series(pd.to_datetime(sequence_main["dates_test"]))

if not validation_df["date"].reset_index(drop=True).equals(validation_sequence_dates):
    raise ValueError("Даты validation для табличного и последовательностного представления не совпали.")
if not test_df["date"].reset_index(drop=True).equals(test_sequence_dates):
    raise ValueError("Даты test для табличного и последовательностного представления не совпали.")

interval_summary = pd.DataFrame(
    [
        get_interval_summary(train_df, "train"),
        get_interval_summary(validation_df, "validation"),
        get_interval_summary(test_df, "test"),
    ]
)

display(Markdown("### Интервалы для основного сценария"))
display(interval_summary)

print(f"Число табличных признаков: {len(feature_columns)}")
print(f"Число последовательностных признаков: {len(sequence_feature_columns)}")
print(f"Размер последовательностных окон: {runtime_config['sequence_window_days']} дней")
print("Табличные признаки:")
print(feature_columns)
print("Последовательностные признаки:")
print(sequence_feature_columns)


## Шаг 4. Метрики и набор сравниваемых моделей

В исследовании используются метрики `MAE`, `RMSE`, `MASE` и `bias`.

- `MAE` удобна как средняя абсолютная ошибка в градусах Цельсия.
- `RMSE` сильнее штрафует крупные промахи.
- `MASE` нормирует ошибку относительно сезонного наивного масштаба и позволяет удобнее сравнивать результаты между сценариями разной длины train.
- `bias` показывает систематическое завышение или занижение прогноза.

`MAPE` здесь не используется как основная метрика, потому что температура в градусах Цельсия может быть близка к нулю или отрицательной, а это делает процентную ошибку плохо интерпретируемой.

Состав моделей:

- `Naive (t-1)`;
- `Seasonal naive (t-365)`;
- `Seasonal AutoReg` как статистический baseline из `statsmodels` с сезонным лагом 365;
- `RandomForestRegressor` как сильная и при этом CPU-friendly табличная ML-модель;
- `Dense-сеть` по лаговым и календарным признакам;
- `GRU` по окнам временного ряда;
- `Conv1D` по тем же окнам временного ряда.

Холт без сезонной компоненты исключён из центрального сравнения: для температурного ряда с выраженной годовой сезонностью такая модель даёт методически слабую базу и на предварительной проверке давала выраженное систематическое смещение.


In [ ]:
def build_tabular_pipeline():
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "regressor",
                RandomForestRegressor(
                    n_estimators=220,
                    max_depth=12,
                    min_samples_leaf=2,
                    random_state=runtime_config["random_state"],
                    n_jobs=-1,
                ),
            ),
        ]
    )


def temporal_tail_split(X, y, validation_share=0.15):
    split_index = int(len(X) * (1 - validation_share))
    if split_index <= 0 or split_index >= len(X):
        raise ValueError("Не удалось выделить внутреннюю валидацию из обучающего интервала.")

    if hasattr(X, "iloc"):
        X_fit, X_valid = X.iloc[:split_index], X.iloc[split_index:]
    else:
        X_fit, X_valid = X[:split_index], X[split_index:]

    if hasattr(y, "iloc"):
        y_fit, y_valid = y.iloc[:split_index], y.iloc[split_index:]
    else:
        y_fit, y_valid = y[:split_index], y[split_index:]

    return X_fit, X_valid, y_fit, y_valid


def prepare_dense_matrices(X_fit, X_valid, X_predict=None):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_fit_ready = scaler.fit_transform(imputer.fit_transform(X_fit))
    X_valid_ready = scaler.transform(imputer.transform(X_valid))
    X_predict_ready = None if X_predict is None else scaler.transform(imputer.transform(X_predict))

    return X_fit_ready, X_valid_ready, X_predict_ready, imputer, scaler


def build_dense_model(input_dim):
    model = keras.Sequential(
        [
            keras.layers.Input(shape=(input_dim,)),
            keras.layers.Dense(48, activation="relu"),
            keras.layers.Dropout(0.15),
            keras.layers.Dense(24, activation="relu"),
            keras.layers.Dense(1),
        ]
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mae",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return model


def build_gru_model(input_shape):
    model = keras.Sequential(
        [
            keras.layers.Input(shape=input_shape),
            keras.layers.GRU(24),
            keras.layers.Dense(12, activation="relu"),
            keras.layers.Dense(1),
        ]
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mae",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return model


def build_conv1d_model(input_shape):
    model = keras.Sequential(
        [
            keras.layers.Input(shape=input_shape),
            keras.layers.Conv1D(24, kernel_size=3, activation="relu", padding="causal"),
            keras.layers.Conv1D(24, kernel_size=3, activation="relu", padding="causal"),
            keras.layers.GlobalAveragePooling1D(),
            keras.layers.Dense(12, activation="relu"),
            keras.layers.Dense(1),
        ]
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mae",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return model


def fit_dense_network(X_fit, y_fit, X_valid, y_valid):
    set_global_seed(runtime_config["random_state"])
    tf.keras.backend.clear_session()

    X_fit_ready, X_valid_ready, _, imputer, scaler = prepare_dense_matrices(X_fit, X_valid)
    model = build_dense_model(X_fit_ready.shape[1])
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=runtime_config["neural_patience"],
            restore_best_weights=True,
        )
    ]
    history = model.fit(
        X_fit_ready,
        np.asarray(y_fit, dtype=float),
        validation_data=(X_valid_ready, np.asarray(y_valid, dtype=float)),
        epochs=runtime_config["neural_epochs"],
        batch_size=runtime_config["neural_batch_size"],
        verbose=0,
        callbacks=callbacks,
    )
    return {
        "model": model,
        "imputer": imputer,
        "scaler": scaler,
        "history": history,
    }


def predict_dense_network(artifacts, X_predict):
    X_ready = artifacts["scaler"].transform(artifacts["imputer"].transform(X_predict))
    return artifacts["model"].predict(X_ready, verbose=0).reshape(-1)


def scale_sequence_arrays(X_fit, X_valid, X_predict=None):
    mean = X_fit.mean(axis=(0, 1), keepdims=True)
    std = X_fit.std(axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)

    X_fit_ready = (X_fit - mean) / std
    X_valid_ready = (X_valid - mean) / std
    X_predict_ready = None if X_predict is None else (X_predict - mean) / std

    return X_fit_ready, X_valid_ready, X_predict_ready, {"mean": mean, "std": std}


def fit_sequence_network(model_builder, X_fit, y_fit, X_valid, y_valid):
    set_global_seed(runtime_config["random_state"])
    tf.keras.backend.clear_session()

    X_fit_ready, X_valid_ready, _, scaling = scale_sequence_arrays(X_fit, X_valid)
    model = model_builder(X_fit_ready.shape[1:])
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=runtime_config["neural_patience"],
            restore_best_weights=True,
        )
    ]
    history = model.fit(
        X_fit_ready,
        np.asarray(y_fit, dtype=float),
        validation_data=(X_valid_ready, np.asarray(y_valid, dtype=float)),
        epochs=runtime_config["neural_epochs"],
        batch_size=runtime_config["neural_batch_size"],
        verbose=0,
        callbacks=callbacks,
    )
    return {
        "model": model,
        "scaling": scaling,
        "history": history,
    }


def predict_sequence_network(artifacts, X_predict):
    mean = artifacts["scaling"]["mean"]
    std = artifacts["scaling"]["std"]
    X_ready = (X_predict - mean) / std
    return artifacts["model"].predict(X_ready, verbose=0).reshape(-1)


def forecast_seasonal_autoreg(history_series, horizon, lags=(1, 2, 3, 7, 14, 30, 365)):
    history_array = np.asarray(history_series, dtype=float)
    effective_lags = [lag for lag in lags if lag < len(history_array) - 1]
    if not effective_lags:
        effective_lags = [1]

    model = AutoReg(history_array, lags=effective_lags, old_names=False, seasonal=False).fit()
    forecast = model.predict(start=len(history_array), end=len(history_array) + horizon - 1, dynamic=False)
    return np.asarray(forecast, dtype=float)


def evaluate_prediction_dict(y_true, prediction_dict, insample):
    metrics = pd.DataFrame(
        [
            evaluate_forecast(y_true, prediction, model_name, insample=insample)
            for model_name, prediction in prediction_dict.items()
        ]
    )
    return metrics.sort_values(["mae", "rmse"]).reset_index(drop=True)


def build_prediction_frame(reference_df, prediction_dict):
    frame = reference_df[["date", "target_tavg", "season"]].reset_index(drop=True).copy()
    for model_name, prediction in prediction_dict.items():
        frame[model_name] = np.asarray(prediction, dtype=float)
    return frame


def run_scenario_experiment(scenario, include_conv1d=False, return_histories=False):
    train_tab, validation_tab, test_tab = split_dataset_by_dates(
        tabular_df,
        train_start=scenario["train_start"],
        validation_start=runtime_config["validation_start_date"],
        validation_end=runtime_config["validation_end_date"],
        test_start=runtime_config["test_start_date"],
        test_end=runtime_config["test_end_date"],
    )

    sequence_split = split_sequence_windows_by_dates(
        X_sequence,
        y_sequence,
        sequence_dates,
        train_start=scenario["train_start"],
        validation_start=runtime_config["validation_start_date"],
        validation_end=runtime_config["validation_end_date"],
        test_start=runtime_config["test_start_date"],
        test_end=runtime_config["test_end_date"],
    )

    validation_dates_sequence = pd.Series(pd.to_datetime(sequence_split["dates_validation"]))
    test_dates_sequence = pd.Series(pd.to_datetime(sequence_split["dates_test"]))
    if not train_tab["date"].is_monotonic_increasing:
        raise ValueError("Train-интервал должен быть отсортирован по времени.")
    if not validation_tab["date"].reset_index(drop=True).equals(validation_dates_sequence):
        raise ValueError(f"В сценарии {scenario['scenario']} не совпали validation-даты.")
    if not test_tab["date"].reset_index(drop=True).equals(test_dates_sequence):
        raise ValueError(f"В сценарии {scenario['scenario']} не совпали test-даты.")

    X_train = train_tab[feature_columns]
    y_train = train_tab["target_tavg"]
    X_validation = validation_tab[feature_columns]
    y_validation = validation_tab["target_tavg"]
    X_test = test_tab[feature_columns]
    y_test = test_tab["target_tavg"]

    validation_predictions = {
        "Naive (t-1)": validation_tab["lag_1"].to_numpy(),
        "Seasonal naive (t-365)": seasonal_naive_forecast(
            validation_tab["date"],
            train_tab.set_index("date")["target_tavg"],
        ).to_numpy(),
        "Seasonal AutoReg": forecast_seasonal_autoreg(y_train, len(validation_tab)),
    }

    print(f"  {scenario['scenario']}: validation -> RandomForest", flush=True)
    tabular_model = build_tabular_pipeline()
    tabular_model.fit(X_train, y_train)
    validation_predictions["RandomForest"] = tabular_model.predict(X_validation)

    print(f"  {scenario['scenario']}: validation -> Dense-сеть", flush=True)
    dense_artifacts = fit_dense_network(X_train, y_train, X_validation, y_validation)
    validation_predictions["Dense-сеть"] = predict_dense_network(dense_artifacts, X_validation)

    print(f"  {scenario['scenario']}: validation -> GRU", flush=True)
    gru_artifacts = fit_sequence_network(
        build_gru_model,
        sequence_split["X_train"],
        sequence_split["y_train"],
        sequence_split["X_validation"],
        sequence_split["y_validation"],
    )
    validation_predictions["GRU"] = predict_sequence_network(gru_artifacts, sequence_split["X_validation"])

    conv_artifacts = None
    if include_conv1d:
        print(f"  {scenario['scenario']}: validation -> Conv1D", flush=True)
        conv_artifacts = fit_sequence_network(
            build_conv1d_model,
            sequence_split["X_train"],
            sequence_split["y_train"],
            sequence_split["X_validation"],
            sequence_split["y_validation"],
        )
        validation_predictions["Conv1D"] = predict_sequence_network(conv_artifacts, sequence_split["X_validation"])

    validation_metrics = evaluate_prediction_dict(y_validation, validation_predictions, insample=y_train)
    validation_metrics["scenario"] = scenario["scenario"]

    full_train_tab = pd.concat([train_tab, validation_tab], ignore_index=True)
    X_full_train = full_train_tab[feature_columns]
    y_full_train = full_train_tab["target_tavg"]

    test_predictions = {
        "Naive (t-1)": test_tab["lag_1"].to_numpy(),
        "Seasonal naive (t-365)": seasonal_naive_forecast(
            test_tab["date"],
            full_train_tab.set_index("date")["target_tavg"],
        ).to_numpy(),
        "Seasonal AutoReg": forecast_seasonal_autoreg(y_full_train, len(test_tab)),
    }

    print(f"  {scenario['scenario']}: test -> RandomForest", flush=True)
    final_tabular_model = build_tabular_pipeline()
    final_tabular_model.fit(X_full_train, y_full_train)
    test_predictions["RandomForest"] = final_tabular_model.predict(X_test)

    X_dense_fit, X_dense_valid, y_dense_fit, y_dense_valid = temporal_tail_split(X_full_train, y_full_train)
    print(f"  {scenario['scenario']}: test -> Dense-сеть", flush=True)
    final_dense_artifacts = fit_dense_network(X_dense_fit, y_dense_fit, X_dense_valid, y_dense_valid)
    test_predictions["Dense-сеть"] = predict_dense_network(final_dense_artifacts, X_test)

    X_sequence_full_train = np.concatenate(
        [sequence_split["X_train"], sequence_split["X_validation"]],
        axis=0,
    )
    y_sequence_full_train = np.concatenate(
        [sequence_split["y_train"], sequence_split["y_validation"]],
        axis=0,
    )
    X_seq_fit, X_seq_valid, y_seq_fit, y_seq_valid = temporal_tail_split(
        X_sequence_full_train,
        y_sequence_full_train,
    )

    print(f"  {scenario['scenario']}: test -> GRU", flush=True)
    final_gru_artifacts = fit_sequence_network(
        build_gru_model,
        X_seq_fit,
        y_seq_fit,
        X_seq_valid,
        y_seq_valid,
    )
    test_predictions["GRU"] = predict_sequence_network(final_gru_artifacts, sequence_split["X_test"])

    final_conv_artifacts = None
    if include_conv1d:
        print(f"  {scenario['scenario']}: test -> Conv1D", flush=True)
        final_conv_artifacts = fit_sequence_network(
            build_conv1d_model,
            X_seq_fit,
            y_seq_fit,
            X_seq_valid,
            y_seq_valid,
        )
        test_predictions["Conv1D"] = predict_sequence_network(final_conv_artifacts, sequence_split["X_test"])

    test_metrics = evaluate_prediction_dict(y_test, test_predictions, insample=y_full_train)
    test_metrics["scenario"] = scenario["scenario"]

    histories = {}
    if return_histories:
        histories = {
            "Dense-сеть": dense_artifacts["history"],
            "GRU": gru_artifacts["history"],
        }
        if include_conv1d and conv_artifacts is not None:
            histories["Conv1D"] = conv_artifacts["history"]

    print(f"  {scenario['scenario']}: сценарий завершён", flush=True)
    return {
        "scenario": scenario,
        "train_frame": train_tab,
        "validation_frame": validation_tab,
        "test_frame": test_tab,
        "validation_metrics": validation_metrics,
        "test_metrics": test_metrics,
        "validation_predictions": build_prediction_frame(validation_tab, validation_predictions),
        "test_predictions": build_prediction_frame(test_tab, test_predictions),
        "histories": histories,
    }


## Шаг 5. Эксперимент: как меняется качество при разной длине train

Ниже все модели сравниваются на одном и том же validation- и test-интервале. Меняется только старт train-периода.

Чтобы не перегружать проект вычислениями, в эксперимент по длине train включены основные модели: `Naive`, `Seasonal naive`, `Seasonal AutoReg`, `RandomForest`, `Dense-сеть` и `GRU`. `Conv1D` оставим для детального сравнения в основном сценарии с максимальной качественной историей.

Для удобства в код добавлены короткие сообщения прогресса: они помогают понять, какой блок сейчас считается, если сценарий идёт несколько минут.


In [ ]:
scenario_order = [
    label
    for label in ["Короткая история", "Средняя история", "Длинная история", "Максимум качественной истории"]
    if label in scenario_lookup
]

scenario_results = {}
for scenario_name in scenario_order:
    scenario = scenario_lookup[scenario_name]
    print(
        "Запуск сценария: "
        f"{scenario_name} ({scenario['train_start'].date()} ... {scenario['train_end'].date()})"
    )
    scenario_results[scenario_name] = run_scenario_experiment(
        scenario,
        include_conv1d=False,
        return_histories=False,
    )

validation_by_train = pd.concat(
    [scenario_results[name]["validation_metrics"] for name in scenario_order],
    ignore_index=True,
)
test_by_train = pd.concat(
    [scenario_results[name]["test_metrics"] for name in scenario_order],
    ignore_index=True,
)

scenario_meta = scenarios_df.set_index("scenario")[["train_start", "train_years_approx"]]
for metrics_frame in [validation_by_train, test_by_train]:
    metrics_frame["train_start"] = metrics_frame["scenario"].map(scenario_meta["train_start"])
    metrics_frame["train_years_approx"] = metrics_frame["scenario"].map(scenario_meta["train_years_approx"])

validation_by_train_path = TABLES_DIR / "02_train_size_validation_metrics.csv"
test_by_train_path = TABLES_DIR / "02_train_size_test_metrics.csv"
validation_by_train.to_csv(validation_by_train_path, index=False)
test_by_train.to_csv(test_by_train_path, index=False)

validation_pivot = (
    validation_by_train.pivot(index="model", columns="scenario", values="mae")
    .reindex(columns=scenario_order)
    .sort_index()
)
test_pivot = (
    test_by_train.pivot(index="model", columns="scenario", values="mae")
    .reindex(columns=scenario_order)
    .sort_index()
)

display(Markdown("### MAE на validation при разной длине train"))
display(validation_pivot.round(3))

display(Markdown("### MAE на test при разной длине train"))
display(test_pivot.round(3))

print(f"Validation-метрики по сценариям сохранены в: {validation_by_train_path}")
print(f"Test-метрики по сценариям сохранены в: {test_by_train_path}")


## Шаг 6. Анализ влияния длины обучающей истории

Отдельно посмотрим, как меняется `MAE` по мере роста train-истории и уменьшается ли разрыв между сильной табличной ML-моделью и лучшими нейросетями.


In [ ]:
train_size_figure_path = FIGURES_DIR / "02_train_size_mae.png"
train_effect_path = TABLES_DIR / "02_train_size_neural_advantage.csv"

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
x_positions = np.arange(len(scenario_order))

for model_name in validation_by_train["model"].unique():
    validation_curve = (
        validation_by_train.loc[validation_by_train["model"] == model_name]
        .set_index("scenario")
        .reindex(scenario_order)
    )
    test_curve = (
        test_by_train.loc[test_by_train["model"] == model_name]
        .set_index("scenario")
        .reindex(scenario_order)
    )

    axes[0].plot(x_positions, validation_curve["mae"], marker="o", label=model_name)
    axes[1].plot(x_positions, test_curve["mae"], marker="o", label=model_name)

axes[0].set_title("MAE на validation при увеличении train")
axes[1].set_title("MAE на test при увеличении train")
for ax in axes:
    ax.set_xticks(x_positions)
    ax.set_xticklabels(scenario_order, rotation=20, ha="right")
    ax.set_ylabel("MAE")
    ax.set_xlabel("Сценарий train")
axes[1].legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(train_size_figure_path, dpi=200, bbox_inches="tight")
plt.show()

train_effect_summary = (
    test_by_train.pivot(index="scenario", columns="model", values="mae")
    .reindex(scenario_order)
    .reset_index()
)
train_effect_summary["Лучшая нейросеть"] = train_effect_summary[["Dense-сеть", "GRU"]].idxmin(axis=1)
train_effect_summary["Лучший нейросетевой MAE"] = train_effect_summary[["Dense-сеть", "GRU"]].min(axis=1)
train_effect_summary["Преимущество лучшей нейросети над RandomForest"] = (
    train_effect_summary["Лучший нейросетевой MAE"] - train_effect_summary["RandomForest"]
)
train_effect_summary.to_csv(train_effect_path, index=False)

display(Markdown("### Как меняется разрыв между лучшей нейросетью и RandomForest на test"))
display(train_effect_summary.round(3))

print(f"График влияния длины train сохранён в: {train_size_figure_path}")
print(f"Таблица преимущества нейросетей сохранена в: {train_effect_path}")


## Шаг 7. Детальное сравнение моделей в основном сценарии

Для главного сравнения берётся сценарий **максимума качественной истории**. Здесь добавляется `Conv1D`, а итоговые таблицы показываются и на validation, и на test.


In [ ]:
main_result = run_scenario_experiment(main_scenario, include_conv1d=True, return_histories=True)

main_validation_metrics = main_result["validation_metrics"].copy()
main_test_metrics = main_result["test_metrics"].copy()

main_validation_path = TABLES_DIR / "02_main_validation_metrics.csv"
main_test_path = TABLES_DIR / "02_main_test_metrics.csv"
main_predictions_path = TABLES_DIR / "02_main_test_predictions.csv"

main_validation_metrics.to_csv(main_validation_path, index=False)
main_test_metrics.to_csv(main_test_path, index=False)
main_result["test_predictions"].to_csv(main_predictions_path, index=False)

display(Markdown("### Основной сценарий: validation"))
display(main_validation_metrics.round(3))

display(Markdown("### Основной сценарий: test"))
display(main_test_metrics.round(3))

print(f"Validation-таблица сохранена в: {main_validation_path}")
print(f"Test-таблица сохранена в: {main_test_path}")
print(f"Таблица прогнозов на test сохранена в: {main_predictions_path}")


## Шаг 8. Дополнительная проверка устойчивости табличной ML-модели

`TimeSeriesSplit` помогает проверить, насколько стабилен `RandomForest` внутри обучающего периода основного сценария. Здесь тоже сохраняется хронологический порядок: каждая следующая валидационная часть идёт позже предыдущей.


In [ ]:
X_train_main = train_df[feature_columns].reset_index(drop=True)
y_train_main = train_df["target_tavg"].reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=5)
cv_records = []

for fold_number, (fit_idx, valid_idx) in enumerate(tscv.split(X_train_main), start=1):
    fold_model = build_tabular_pipeline()
    fold_model.fit(X_train_main.iloc[fit_idx], y_train_main.iloc[fit_idx])
    fold_prediction = fold_model.predict(X_train_main.iloc[valid_idx])

    fold_metrics = evaluate_forecast(
        y_train_main.iloc[valid_idx],
        fold_prediction,
        model_name=f"fold_{fold_number}",
        insample=y_train_main.iloc[fit_idx],
    )
    fold_metrics["rows"] = len(valid_idx)
    cv_records.append(fold_metrics)

cv_results = pd.DataFrame(cv_records)
cv_results_path = TABLES_DIR / "02_random_forest_timeseriessplit.csv"
cv_results.to_csv(cv_results_path, index=False)

display(cv_results.round(3))
print(f"Таблица TimeSeriesSplit сохранена в: {cv_results_path}")


## Шаг 9. Графики прогнозов, метрик и кривых обучения

Ниже визуализируются результаты основного сценария. Это помогает увидеть не только итоговые числа метрик, но и характер ошибок на всём тестовом интервале.


In [ ]:
prediction_figure_path = FIGURES_DIR / "02_main_test_predictions.png"
metrics_figure_path = FIGURES_DIR / "02_main_test_metrics.png"
learning_curves_path = FIGURES_DIR / "02_neural_learning_curves.png"

main_predictions_test = main_result["test_predictions"].copy()

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(main_predictions_test["date"], main_predictions_test["target_tavg"], label="Факт", color="black", linewidth=2)
for model_name in [column for column in main_predictions_test.columns if column not in {"date", "target_tavg", "season"}]:
    ax.plot(main_predictions_test["date"], main_predictions_test[model_name], label=model_name, linewidth=1.1, alpha=0.9)
ax.set_title("Реальные и предсказанные значения на test")
ax.set_xlabel("Дата")
ax.set_ylabel("Температура, °C")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(prediction_figure_path, dpi=200, bbox_inches="tight")
plt.show()

metric_columns = ["mae", "rmse", "mase"]
x = np.arange(len(main_test_metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for shift, metric_name in enumerate(metric_columns):
    ax.bar(x + (shift - 1) * width, main_test_metrics[metric_name], width=width, label=metric_name.upper())
ax.set_xticks(x)
ax.set_xticklabels(main_test_metrics["model"], rotation=20, ha="right")
ax.set_title("Сравнение ключевых метрик на test")
ax.set_ylabel("Значение метрики")
ax.legend()
fig.tight_layout()
fig.savefig(metrics_figure_path, dpi=200, bbox_inches="tight")
plt.show()

neural_histories = main_result["histories"]
fig, axes = plt.subplots(1, len(neural_histories), figsize=(5 * len(neural_histories), 4), squeeze=False)
axes = axes.ravel()
for ax, (model_name, history) in zip(axes, neural_histories.items()):
    ax.plot(history.history.get("loss", []), label="train MAE")
    ax.plot(history.history.get("val_loss", []), label="validation MAE")
    ax.set_title(model_name)
    ax.set_xlabel("Эпоха")
    ax.set_ylabel("MAE")
    ax.legend()
fig.tight_layout()
fig.savefig(learning_curves_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"График прогнозов сохранён в: {prediction_figure_path}")
print(f"График метрик сохранён в: {metrics_figure_path}")
print(f"Кривые обучения сохранены в: {learning_curves_path}")


## Шаг 10. Анализ ошибок по сезонам

Сезонный анализ помогает ответить на два исследовательских вопроса:

- в какие времена года модели ошибаются сильнее;
- у каких подходов заметнее систематическое смещение.


In [ ]:
seasonal_records = []
for model_name in [column for column in main_predictions_test.columns if column not in {"date", "target_tavg", "season"}]:
    seasonal_records.append(
        pd.DataFrame(
            {
                "date": main_predictions_test["date"],
                "season": main_predictions_test["season"],
                "model": model_name,
                "absolute_error": (main_predictions_test[model_name] - main_predictions_test["target_tavg"]).abs(),
                "residual": main_predictions_test[model_name] - main_predictions_test["target_tavg"],
            }
        )
    )

seasonal_error_df = pd.concat(seasonal_records, ignore_index=True)
seasonal_summary = seasonal_error_df.groupby(["model", "season"]).agg(
    mae_by_season=("absolute_error", "mean"),
    bias_by_season=("residual", "mean"),
    rows=("absolute_error", "size"),
).reset_index()

seasonal_path = TABLES_DIR / "02_seasonal_error_analysis.csv"
seasonal_summary.to_csv(seasonal_path, index=False)

display(Markdown("### Таблица сезонных ошибок"))
display(seasonal_summary.round(3))

season_order = ["зима", "весна", "лето", "осень"]
season_figure_path = FIGURES_DIR / "02_error_by_season.png"
pivot_mae = seasonal_summary.pivot(index="season", columns="model", values="mae_by_season").reindex(season_order)

fig, ax = plt.subplots(figsize=(12, 5))
for model_name in pivot_mae.columns:
    ax.plot(pivot_mae.index, pivot_mae[model_name], marker="o", label=model_name)
ax.set_title("MAE по сезонам на test")
ax.set_xlabel("Сезон")
ax.set_ylabel("MAE")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(season_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Сезонная таблица сохранена в: {seasonal_path}")
print(f"График сезонных ошибок сохранён в: {season_figure_path}")


In [ ]:
best_validation_model = main_validation_metrics.iloc[0]
best_test_model = main_test_metrics.iloc[0]
best_neural_test = main_test_metrics.loc[
    main_test_metrics["model"].isin(["Dense-сеть", "GRU", "Conv1D"])
].sort_values(["mae", "rmse"]).iloc[0]
histgb_test = main_test_metrics.loc[main_test_metrics["model"] == "RandomForest"].iloc[0]

short_advantage = float(
    train_effect_summary.loc[
        train_effect_summary["scenario"] == scenario_order[0],
        "Преимущество лучшей нейросети над RandomForest",
    ].iloc[0]
)
long_advantage = float(
    train_effect_summary.loc[
        train_effect_summary["scenario"] == scenario_order[-1],
        "Преимущество лучшей нейросети над RandomForest",
    ].iloc[0]
)

if long_advantage < 0 and short_advantage < 0:
    train_effect_comment = (
        "На всех сценариях лучшая нейросеть обгоняет RandomForest по MAE. "
        "Это означает, что при увеличении train-истории преимущество нейросетей сохраняется."
    )
elif long_advantage > 0 and short_advantage > 0:
    train_effect_comment = (
        "Во всех сценариях RandomForest остаётся сильнее лучших нейросетей по MAE. "
        "Значит, увеличение train-истории не перевернуло лидерство, хотя абсолютная ошибка моделей менялась."
    )
else:
    train_effect_comment = (
        "Относительное преимущество нейросетей меняется по мере роста train-истории: "
        "на части сценариев они лучше RandomForest, а на части уступают ему."
    )

if best_neural_test["mae"] < histgb_test["mae"]:
    neural_vs_ml_comment = (
        f"Лучшая нейросеть на test — **{best_neural_test['model']}** — обошла RandomForest "
        f"по MAE ({best_neural_test['mae']:.3f} против {histgb_test['mae']:.3f})."
    )
else:
    neural_vs_ml_comment = (
        f"Лучшая нейросеть на test — **{best_neural_test['model']}** — не смогла обойти RandomForest "
        f"по MAE ({best_neural_test['mae']:.3f} против {histgb_test['mae']:.3f})."
    )

summary_text = f"""
## Выводы

Лучшая модель на validation в основном сценарии: **{best_validation_model['model']}**.

Лучшая модель на фиксированном test-интервале: **{best_test_model['model']}**.

- {neural_vs_ml_comment}
- Seasonal naive полезен как жёсткий ориентир для сезонного ряда, но он не учитывает внутрисезонные колебания и обычно уступает более сильным моделям.
- `Seasonal AutoReg` даёт более содержательный статистический baseline, потому что использует как короткие лаги, так и сезонный лаг порядка года.
- {train_effect_comment}
- Даже если нейросети не становятся абсолютными лидерами, это нормальный исследовательский результат: для хорошо подготовленного табличного ряда сильная ML-модель может конкурировать с компактными нейросетями, а преимущество архитектур зависит от объёма истории и способа представления данных.
- Сезонный анализ показывает, что качество модели полезно оценивать не только по одной усреднённой метрике: ошибки могут заметно отличаться между зимой, летом и переходными сезонами.
"""

display(Markdown(summary_text))
